# [Chapter 19 — HIV, §19.7] Perinatal HIV: Mother-to-Child Transmission Intervention Coverage

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 19 — HIV
**Considerations developed:** 12 (basic vs effective $R$)
**Estimated runtime:** ≤ 15 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_19_hiv/03_perinatal_HIV_intervention.ipynb)


## What this notebook does

Models the impact of antenatal antiretroviral coverage on mother-to-child HIV transmission. With untreated MTCT probability $p_v \approx 0.30$ and on-treatment $p_v \approx 0.02$, expanding coverage from 0% to 90% reduces effective MTCT probability from 0.30 to 0.04 — a 7.5x reduction. Demonstrates how Consideration~12 applies even to a *parameter* (not a $\mathcal{R}$) by showing that the headline statistic $p_v$ misrepresents the actual situation when intervention coverage varies.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import csv, json
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_19
from shared.verification import assert_within_tolerance
set_seed_chapter_19()
book_style()

# Path to the synthetic-data root for the case studies
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'data'))


## Effective MTCT probability vs intervention coverage

In [ ]:
from shared.parameters import baseline_chapter_19
p = baseline_chapter_19()
p_untreated = p['p_v']
p_treated = p['p_v_treated']
print(f'p_v untreated:  {p_untreated:.2f}')
print(f'p_v on therapy: {p_treated:.2f}')
print(f'Within-individual reduction: {(1 - p_treated/p_untreated)*100:.0f}%')

coverage_grid = np.linspace(0, 1, 21)
p_v_effective = (1 - coverage_grid)*p_untreated + coverage_grid*p_treated

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(coverage_grid*100, p_v_effective*100, 'o-', color=BOOK_COLORS['primary'], lw=1.8)
ax.axhline(p_untreated*100, ls=':', color=BOOK_COLORS['secondary'], label='untreated p_v')
ax.axhline(p_treated*100, ls=':', color=BOOK_COLORS['highlight'], label='on-therapy p_v')
ax.set_xlabel('Antenatal ART coverage (%)')
ax.set_ylabel('Effective MTCT probability (%)')
ax.set_title('Coverage-modulated MTCT: headline parameter masks intervention impact')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

for cov in [0, 0.5, 0.9, 0.99]:
    eff = (1 - cov)*p_untreated + cov*p_treated
    print(f'Coverage {cov:.0%}:  effective p_v = {eff:.4f}  ({1/eff:.0f}x reduction in offspring at risk)')

# Verify
eff_full = (1 - 0.9)*p_untreated + 0.9*p_treated
assert eff_full < 0.05, '90% coverage should reduce effective p_v below 5%'
print('\nVerified: 90% coverage reduces effective MTCT below 5%.')


## Population-level R_0 contribution from vertical transmission

Even with low effective $p_v$, vertical transmission contributes additively to $\mathcal{R}_0$ (Chapter 15 framework). For human demographics (40-year reproductive lifespan), the contribution is small but non-zero, and intervention scaling shifts the threshold.

In [ ]:
# Approximate vertical contribution to R_0:
# R_0_vert ~ 2 * p_v_effective (each infected woman has ~2 children on average)
n_children_avg = 2.0
R0_vert_contribution = n_children_avg * p_v_effective

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(coverage_grid*100, R0_vert_contribution, 'o-', color=BOOK_COLORS['primary'], lw=1.8)
ax.set_xlabel('ART coverage (%)')
ax.set_ylabel('R_0 contribution from vertical route')
ax.set_title('Vertical-route $R_0$ contribution under intervention scaling')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print(f'R_0_vert at 0% coverage:  {R0_vert_contribution[0]:.3f}')
print(f'R_0_vert at 90% coverage: {R0_vert_contribution[18]:.3f}')
print(f'\nNote: this is the within-cohort contribution, not the bipartite-cycle R_0.')


## What's next

- Combined sexual + vertical transmission requires the full Chapter 15 framework with bipartite cycle.
- The on-therapy MTCT probability of ~2% reflects modern combination ART regimens; older single-drug regimens (e.g., short-course nevirapine) achieved ~10%, which still represented a major public-health win.
- The same coverage-vs-effectiveness trade-off appears in TB preventive therapy, malaria seasonal chemoprevention, and any biomedical intervention with imperfect within-individual efficacy.